# **VLSI 2026 Code-a-Chip Challenge**
## **ROAR (Robust Optimal Analog Reuse) Showcase and Demonstration: <br><br> A GUI Based Tool for Analog Design Automation Using The gm/ID and C/ID Methodologies**

## Design Examples From Specification to Layout for Common Source Amplifier, Current Mirror OTA, and Discrete Time Comparator Done With Both The Skywater 130nm and IHP 130nm PDKs

## **Author: Alec S. Adair - The University of Utah (The U of U), Salt Lake City**
**alecadair1@gmail.com, alec.adair@utah.edu, https://github.com/alecadair, https://www.linkedin.com/in/alecadair/**
## With support from Dr. Armin Tajalli and LCAS (Laboratory for Circuits and Systems) at The U of U
**https://lcas.ece.utah.edu**

Please feel free to reach out!

**Work Licensed Under Apache 2.0**

<div style="display:flex; justify-content:center; align-items:center; gap:24px;">
  <img src="images/png/ROAR_LOGO.png" alt="ROAR LOGO" style="width:55%; max-width:420px; height:auto;">
  <img src="images/png/lcas_logo_v0.png" alt="LCAS LOGO" style="width:55%; max-width:420px; height:auto;">
</div>

---

### Abstract
ROAR (Robust and Optimal Analog Reuse) is an open-source, GUI-based analog integrated circuit (IC) design automation tool that supports lookup table (LUT)–driven workflows in a graphical, PDK-agnostic framework. ROAR emphasizes the **gm/ID** and **C/ID** methodologies, using engineer-derived analytical design equations to help decouple transistor sizing and design optimization from any single process technology.

Traditional LUT-based design flows often require substantial scripting and bespoke data handling—work that is time-consuming and error-prone, and not typically the primary focus of many analog design engineers. This burden grows significantly when designing for robustness across process, voltage, and temperature (PVT) corners, where many corner-specific lookup tables must be generated, organized, and queried. Traditionally maintaining scalable workflows therefore demands meticulous file management and carefully structured code to keep results consistent and reproducible.

This tool is designed to remove this overhead by providing a GUI-driven environment with clear LUT corner selection, fast, snappy, and strong industry standard graphing capabilities, along with simple design equation input and lookup capabilities. ROAR also has the capability to generate and run design scripts from the GUI with an integrated Python console and text editor.

ROAR has been designed and built to be simple and to allow designers to rapidly visualize design spaces, identify robust regions across PVT, and efficiently navigate solution spaces to form clear paths of design convergence.

This notebook demonstrates the complete ROAR design flow:

1. **LUT Characterization** — How transistor data is extracted and organized  
2. **Device Exploration** — Visualizing gm/ID design curves across PVT corners  
3. **Analytical Equation-Based Design** — Sizing a current-mirror OTA with symbolic expressions  
5. **Multi-Corner Analysis** — Ensuring robust designs across process, voltage, and temperature

All examples in this notebook use the **SkyWater 130nm** and **IHP 130nm** open PDKs with 9 PVT corners (SS/TT/FF × −25 °C/25 °C/75 °C).

## **General Flow**

<p align="center">
  <img src="images/png/design_flowchart.png" alt="Flow Diagram" style="width: 65%;">
</p>

---
## 1. Download and Install ROAR
Follow these instructions to quickly download, install, and run ROAR with Skywater and IHP 130 PDK lookup tables pre-loaded in.  
PDKs are characterized for many device lengths at -25, 25, and 75 Celsius temperatures across corners.
### 1.1 Clone ROAR from git repository located at https://github.com/alecadair/roar


In [18]:
#!git clone https://github.com/alecadair/roar

### 1.2 Change working directory to top level roar directory and run make to install


In [19]:
#!cd roar && make

### 1.3 Set environmental variables needed by ROAR by sourcing roar_env.csh file (generated from make command)


In [20]:
#!source roar_env.csh

### 1.4 Run ROAR with any version of Python 3

In [21]:
#!python3 src/gui/roar_gui.py

### Congratulations you have successfully downloaded, installed, and launched ROAR!

---
## 2. GUI Components
The purpose of this section is to give a high level demonstration of the different components and functionalities within the ROAR software. When running ROAR a GUI should pop that looks like this:

---
## 2. Technology and Device Exploration and Comparison
One of the key value propositions of ROar is the ability to very quickly and graphically analyze and compare devices across technologies. For this demonstration similar devices in both the IHP and Skywater 130 PDKs will be explored to show their strengths, weaknesses, and also somewhat peculiar models.  

To start this exploration It is recommended to expand one o

---
## 1. Background: The gm/ID Design Methodology

The **gm/ID** (also called *inverse-ID* or C/ID) methodology, popularized by Silveira, Flandre, Jespers, and Murmann, provides a unified framework for analog transistor sizing that works seamlessly from **weak inversion** through **strong inversion** — unlike the classical square-law model which breaks down in modern short-channel processes.

### Key Idea

Instead of relying on analytical MOSFET equations, the designer works with **measured or simulated** device characteristics, parameterized by the *transconductance efficiency* **gm/ID** (denoted `kgm` in ROAR):

$$\frac{g_m}{I_D} \quad \text{[V}^{-1}\text{]}$$

This single variable spans the entire operating region:
- **High gm/ID** (> 20 V⁻¹) → Weak inversion → Low power, low speed  
- **Low gm/ID** (< 10 V⁻¹) → Strong inversion → High speed, high power  

All relevant design parameters — $f_T$, $g_m \cdot r_o$, capacitance ratios, current density — are uniquely determined by gm/ID for a given device, channel length, and PVT corner. This creates a complete **design space** that ROAR visualizes and optimizes over.

### References

- P. Jespers and B. Murmann, *Systematic Design of Analog CMOS Circuits Using Pre-Computed Lookup Tables*, Cambridge University Press, 2017.  
- B. Murmann, "The gm/ID Design Methodology: A Tutorial," *IEEE Solid-State Circuits Magazine*, 2023.

---
## 2. Environment Setup

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import EngFormatter
import warnings
warnings.filterwarnings('ignore')

# Set up ROAR paths
ROAR_HOME = os.path.abspath('.')
os.environ['ROAR_HOME'] = ROAR_HOME
sys.path.insert(0, os.path.join(ROAR_HOME, 'src', 'gui'))
sys.path.insert(0, os.path.join(ROAR_HOME, 'src'))

# Import ROAR core modules (headless — no PyQt6 required)
# Use Agg backend to avoid display issues in notebook
matplotlib.use('Agg')
%matplotlib inline

from cid import CIDCorner, CIDDevice
from equation_solver import ROAREquationSolver

# Plotting style
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
    'svg.fonttype': 'none',
})

# Color palette for PVT corners
CORNER_COLORS = {
    'ss-25': '#1f77b4', 'ss25': '#2ca02c', 'ss75': '#d62728',
    'tt-25': '#9467bd', 'tt25': '#8c564b', 'tt75': '#e377c2',
    'ff-25': '#7f7f7f', 'ff25': '#bcbd22', 'ff75': '#17becf',
}

print(f"ROAR_HOME: {ROAR_HOME}")
print(f"NumPy:     {np.__version__}")
print(f"Pandas:    {pd.__version__}")
print("ROAR core modules loaded successfully (headless mode).")

---
## 3. LUT Characterization Flow

ROAR's design methodology starts with **characterization**: extracting MOSFET small-signal parameters from SPICE simulation and storing them in CSV lookup tables.

### 3.1 Characterization Procedure

A **diode-connected** MOSFET test bench is used (VGS = VDS). A drain current is swept logarithmically (e.g., 1 nA to 1 mA, 20 points/decade), and at each bias point the simulator extracts:

| Parameter | Description | Units |
|-----------|-------------|-------|
| `gm` | Transconductance | S |
| `gds` | Output conductance | S |
| `cgg, cgs, cgd, cdd, css, cds` | Capacitances | F |
| `vth, vdsat, vgs, vds` | Voltages | V |
| `kgm` (gm/ID) | Transconductance efficiency | V⁻¹ |

This is repeated for every combination of **device model** × **channel length** × **PVT corner**, producing a structured directory tree:

```
LUTs_SKY130/
├── n_01v8/                    # NMOS regular Vt
│   ├── LUT_N_150/             # L = 150 nm
│   │   ├── nfettt27.csv       # Typical-Typical, 27°C
│   │   ├── nfetss-25.csv      # Slow-Slow, -25°C
│   │   ├── nfetff75.csv       # Fast-Fast, 75°C
│   │   └── ... (9 corners)
│   ├── LUT_N_500/             # L = 500 nm
│   └── ...
├── p_01v8/                    # PMOS regular Vt
├── n_01v8_lvt/                # NMOS low Vt
└── p_01v8_lvt/                # PMOS low Vt
```

### 3.2 PDK Agnosticism

ROAR's LUT format is **PDK-agnostic**. Any simulator (ngspice, Spectre, HSPICE, Xyce) that can produce the required CSV columns can generate compatible LUTs. ROAR currently ships with pre-characterized LUTs for:

- **SkyWater SKY130** (open-source 130 nm)
- **IHP SG13G2** (open-source 130 nm BiCMOS)

New PDKs are registered by simply adding one line to `tech_list.txt`.

### 3.3 Loading and Inspecting a LUT

In [ ]:
# Load a single LUT CSV and inspect its structure
lut_path = os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130',
                        'n_01v8', 'LUT_N_500', 'nfettt25.csv')

df_example = pd.read_csv(lut_path, skipinitialspace=True)
df_example.columns = df_example.columns.str.lower()

print(f"LUT: {os.path.basename(lut_path)}")
print(f"PDK: {df_example['pdk'].iloc[0]}")
print(f"W = {df_example['w'].iloc[0]} µm,  L = {df_example['l'].iloc[0]} µm")
print(f"Number of bias points: {len(df_example)}")
print(f"\nColumns ({len(df_example.columns)}):")
print(list(df_example.columns))
print(f"\nDrain current range: {df_example['id'].min():.2e} A  →  {df_example['id'].max():.2e} A")
print(f"gm/ID range:         {df_example['kgm'].min():.1f}  →  {df_example['kgm'].max():.1f} V⁻¹")

# Show first few rows of key columns
key_cols = ['id', 'gm', 'kgm', 'gds', 'cgg', 'cgs', 'cgd', 'cdd', 'ft', 'vth', 'vdsat', 'vgs']
available_cols = [c for c in key_cols if c in df_example.columns]
df_example[available_cols].head(10)

---
## 4. Device Exploration — Visualizing the gm/ID Design Space

ROAR's `CIDCorner` and `CIDDevice` classes automatically compute derived parameters (fₜ, gm·rₒ, current density, capacitance ratios) upon loading a LUT CSV. This creates a rich design space that the engineer can explore.

### 4.1 Loading All Corners for a Device

In [ ]:
def load_device_corners(model_dir, length_dir):
    """Load all PVT corner CSVs for a given model and channel length."""
    lut_base = os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130')
    lut_dir = os.path.join(lut_base, model_dir, length_dir)
    
    corners = {}
    for csv_file in sorted(os.listdir(lut_dir)):
        if csv_file.endswith('.csv'):
            corner_name = csv_file.replace('.csv', '')
            csv_path = os.path.join(lut_dir, csv_file)
            corner = CIDCorner(corner_name=corner_name, lut_csv=csv_path, vdd=1.8)
            corners[corner_name] = corner
    return corners

# Load NMOS L=500nm (used in the CM OTA design) — all 9 PVT corners
nmos_500_corners = load_device_corners('n_01v8', 'LUT_N_500')

# Load PMOS L=500nm — all 9 PVT corners
pmos_500_corners = load_device_corners('p_01v8', 'LUT_P_500')

print(f"Loaded {len(nmos_500_corners)} NMOS corners: {list(nmos_500_corners.keys())}")
print(f"Loaded {len(pmos_500_corners)} PMOS corners: {list(pmos_500_corners.keys())}")

### 4.2 Canonical gm/ID Design Curves

The four canonical plots of the gm/ID methodology reveal the fundamental design trade-offs:

1. **gm/ID vs. Current Density** — Maps operating region to bias current  
2. **fₜ vs. gm/ID** — Speed vs. power efficiency trade-off  
3. **gm·rₒ vs. gm/ID** — Intrinsic gain vs. power efficiency  
4. **(gm/ID)·fₜ vs. gm/ID** — Combined gain-bandwidth figure of merit

In [ ]:
def plot_four_canonical(corners, device_label, figsize=(14, 10)):
    """Plot the four canonical gm/ID design curves across all PVT corners."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    fig.suptitle(f'SKY130 {device_label} — gm/ID Design Space (9 PVT Corners)',
                 fontsize=14, fontweight='bold')
    
    plot_configs = [
        ('kgm', 'iden',  'gm/ID [V⁻¹]', 'Current Density ID/W [A/µm]',
         'gm/ID vs. Current Density', False, True),
        ('kgm', 'ft',    'gm/ID [V⁻¹]', 'Transit Frequency fₜ [Hz]',
         'fₜ vs. gm/ID', False, True),
        ('kgm', 'gmro',  'gm/ID [V⁻¹]', 'Intrinsic Gain gm·rₒ [V/V]',
         'gm·rₒ vs. gm/ID', False, True),
        ('kgm', 'kgmft', 'gm/ID [V⁻¹]', '(gm/ID)·fₜ [Hz/V]',
         '(gm/ID)·fₜ vs. gm/ID', False, True),
    ]
    
    colors = list(CORNER_COLORS.values())
    
    for ax, (x_param, y_param, xlabel, ylabel, title, xlog, ylog) in zip(axes.flat, plot_configs):
        for i, (corner_name, corner) in enumerate(corners.items()):
            df = corner.df
            # Filter to valid gm/ID range (0.5 to 30)
            mask = (df['kgm'] > 0.5) & (df['kgm'] < 30)
            x_data = df.loc[mask, x_param].values
            y_data = df.loc[mask, y_param].values
            
            ax.plot(x_data, y_data, color=colors[i % len(colors)],
                    label=corner_name, linewidth=1.2, alpha=0.85)
        
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontsize=11)
        if xlog: ax.set_xscale('log')
        if ylog: ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
    
    # Single shared legend
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=5,
               fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()

# NMOS canonical curves
plot_four_canonical(nmos_500_corners, 'NMOS n_01v8 (L = 500 nm)')

In [ ]:
# PMOS canonical curves
plot_four_canonical(pmos_500_corners, 'PMOS p_01v8 (L = 500 nm)')

### 4.3 Process Variation Analysis

A key strength of the gm/ID methodology is the ability to **quantify** how PVT variation affects every design parameter. Let's examine the spread of intrinsic gain and transit frequency across corners.

In [ ]:
# Compare fT and gmro at a specific gm/ID operating point across corners
kgm_targets = [5, 10, 15, 20, 25]  # V^-1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for device_type, corners, ax, param, ylabel, title in [
    ('NMOS', nmos_500_corners, ax1, 'ft', 'fₜ [GHz]', 'NMOS fₜ Across PVT Corners'),
    ('NMOS', nmos_500_corners, ax2, 'gmro', 'gm·rₒ [V/V]', 'NMOS gm·rₒ Across PVT Corners'),
]:
    corner_names = list(corners.keys())
    data = {kgm: [] for kgm in kgm_targets}
    
    for corner_name, corner in corners.items():
        for kgm_t in kgm_targets:
            val = corner.lookup('kgm', param, kgm_t)
            scale = 1e-9 if param == 'ft' else 1.0
            data[kgm_t].append(val * scale)
    
    x = np.arange(len(corner_names))
    width = 0.15
    for j, kgm_t in enumerate(kgm_targets):
        offset = (j - len(kgm_targets)/2) * width + width/2
        bars = ax.bar(x + offset, data[kgm_t], width, label=f'gm/ID = {kgm_t}', alpha=0.85)
    
    ax.set_xlabel('PVT Corner')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(corner_names, rotation=45, ha='right', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 4.4 Channel Length Comparison

ROAR's LUTs span multiple channel lengths. Let's visualize how the gm·rₒ vs. gm/ID trade-off changes with length.

In [ ]:
# Compare NMOS gm·ro across channel lengths (TT, 25°C corner)
lengths = ['LUT_N_150', 'LUT_N_200', 'LUT_N_250', 'LUT_N_300', 'LUT_N_500', 'LUT_N_1000']
length_labels = ['150 nm', '200 nm', '250 nm', '300 nm', '500 nm', '1000 nm']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cmap = plt.cm.viridis(np.linspace(0, 0.9, len(lengths)))

for i, (length_dir, label) in enumerate(zip(lengths, length_labels)):
    lut_path = os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130',
                            'n_01v8', length_dir, 'nfettt25.csv')
    if os.path.exists(lut_path):
        corner = CIDCorner(corner_name=f'tt25_{label}', lut_csv=lut_path, vdd=1.8)
        df = corner.df
        mask = (df['kgm'] > 0.5) & (df['kgm'] < 30)
        
        ax1.semilogy(df.loc[mask, 'kgm'], df.loc[mask, 'gmro'],
                     color=cmap[i], linewidth=2, label=label)
        ax2.semilogy(df.loc[mask, 'kgm'], df.loc[mask, 'ft'],
                     color=cmap[i], linewidth=2, label=label)

ax1.set_xlabel('gm/ID [V⁻¹]')
ax1.set_ylabel('Intrinsic Gain gm·rₒ [V/V]')
ax1.set_title('NMOS Intrinsic Gain vs. Channel Length\n(TT, 25°C)', fontweight='bold')
ax1.legend(title='Channel Length', fontsize=9)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('gm/ID [V⁻¹]')
ax2.set_ylabel('Transit Frequency fₜ [Hz]')
ax2.set_title('NMOS Transit Frequency vs. Channel Length\n(TT, 25°C)', fontweight='bold')
ax2.legend(title='Channel Length', fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Equation-Based Design: Current-Mirror OTA

ROAR's **Design Editor** allows the designer to express analog design equations symbolically. Expressions can reference **LUT lookups** (e.g., `kgm:M1_2` reads the gm/ID column from the LUT assigned to instance M1_2) and are automatically resolved using topological sorting of the dependency graph.

### 5.1 Circuit Architecture

We design a **single-stage current-mirror OTA** (folded architecture), a workhorse topology for many analog applications. The circuit has four transistor groups:

| Instance | Role | Type | Sizing |
|----------|------|------|--------|
| **M1_2** | Input diff pair | NMOS (n_01v8) | L = 500 nm |
| **M3_4** | Diode-connected loads | PMOS (p_01v8) | L = 500 nm |
| **M5_6** | Current mirrors (NMOS) | NMOS (n_01v8) | L = 500 nm |
| **M7_8** | Current mirrors (PMOS) | PMOS (p_01v8) | L = 500 nm |

### 5.2 Design Specifications

| Parameter | Value |
|-----------|-------|
| Unity-gain frequency (fᵤ) | 100 MHz |
| Load capacitance (C_L) | 10 pF |
| Voltage gain (Aᵥ) | ≥ 30 dB |
| Supply voltage (VDD) | 1.8 V |

In [ ]:
# Load the CM OTA design JSON (same file the ROAR GUI saves/loads)
with open(os.path.join(ROAR_HOME, 'design', 'current_mirror_ota.json'), 'r') as f:
    cm_ota_design = json.load(f)

print("=" * 70)
print("CURRENT-MIRROR OTA — Design Equations")
print("=" * 70)
print(f"{'Symbol':<20} {'Expression':<50}")
print("-" * 70)
for eq in cm_ota_design['expression_editor']:
    sym = eq['Symbol']
    expr = eq['Expression']
    print(f"{sym:<20} {expr:<50}")

print("\n" + "=" * 70)
print("Instance Table")
print("=" * 70)
print(f"{'Instance':<12} {'Corners':<50}")
print("-" * 70)
for inst in cm_ota_design['instance_table']:
    print(f"{inst['Instance']:<12} {inst['Corners']:<50}")

### 5.3 Setting Up the Equation Solver Programmatically

We now replicate what the ROAR GUI does: instantiate the `ROAREquationSolver`, load corner DataFrames, add the design equations, and evaluate them across all sweep points and PVT corners.

In [ ]:
def build_corner_dfs(design_json):
    """Build the device-namespaced corner DataFrames dict from a design JSON.
    
    Returns a dict with keys like 'M1_2::PDK>SKY130A>n_01v8>500>nfettt75'
    mapping to pandas DataFrames.
    """
    lut_base = os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130')
    corner_dfs = {}
    
    for inst in design_json['instance_table']:
        instance_name = inst['Instance']
        for corner_data in inst.get('_corners_data', []):
            model = corner_data['model']
            length = corner_data['length']
            name = corner_data['name']
            path_key = corner_data['path']
            
            # Determine N or P from the model name
            if 'n_' in model:
                prefix = 'N'
            else:
                prefix = 'P'
            
            lut_dir = f'LUT_{prefix}_{length}'
            csv_path = os.path.join(lut_base, model, lut_dir, f'{name}.csv')
            
            if os.path.exists(csv_path):
                corner = CIDCorner(corner_name=name, lut_csv=csv_path, vdd=1.8)
                ns_key = f"{instance_name}::{path_key}"
                corner_dfs[ns_key] = corner.df
    
    return corner_dfs


def setup_solver(design_json):
    """Create and configure a ROAREquationSolver from a design JSON."""
    corner_dfs = build_corner_dfs(design_json)
    
    # Create solver
    solver = ROAREquationSolver(top_level_app=None)
    
    # Add equations from the design JSON
    symbols_to_add = []
    for eq in design_json['expression_editor']:
        if not eq.get('disabled', False):
            sym = eq['Symbol']
            expr = eq['Expression']
            error = solver.add_equation(sym, expr)
            if error:
                print(f"  Warning: {sym} = {expr} → {error}")
            symbols_to_add.append(sym)
    
    return solver, corner_dfs, symbols_to_add

# Build the solver for the CM OTA
solver, corner_dfs, symbols = setup_solver(cm_ota_design)

print(f"Loaded {len(corner_dfs)} corner DataFrames")
print(f"Added {len(solver.equations)} equations")
print(f"\nCorner DataFrame keys (first 6):")
for k in list(corner_dfs.keys())[:6]:
    print(f"  {k}")

In [ ]:
# Evaluate all design equations
results = solver.evaluate_equations(symbols_to_add=symbols, corner_dfs=corner_dfs)

if results is not None:
    print("✓ Equations evaluated successfully!\n")
    print(f"{'Symbol':<18} {'Shape':<20} {'Min':<15} {'Max':<15}")
    print("-" * 68)
    for sym in symbols:
        val = results.get(sym)
        if val is not None:
            if isinstance(val, np.ndarray):
                shape_str = str(val.shape)
                # Filter out non-finite values for display
                finite_vals = val[np.isfinite(val)]
                if len(finite_vals) > 0:
                    min_str = f"{np.min(finite_vals):.4e}"
                    max_str = f"{np.max(finite_vals):.4e}"
                else:
                    min_str = max_str = "N/A"
            else:
                shape_str = "scalar"
                min_str = max_str = f"{val:.4e}" if isinstance(val, float) else str(val)
            print(f"{sym:<18} {shape_str:<20} {min_str:<15} {max_str:<15}")
else:
    print("✗ Equation evaluation failed.")

### 5.4 Visualizing Design Trade-Offs

ROAR enables the designer to plot any computed expression against any other, revealing the design trade-off space. The plots below show the **drain current**, **voltage gain**, and **GBW** as functions of the gm/ID operating point, evaluated across all PVT corners simultaneously.

In [ ]:
def plot_design_results(results, x_sym, y_sym, xlabel, ylabel, title,
                        ax=None, yscale='linear', constraint_line=None,
                        constraint_label=None, positive_only=True):
    """Plot a design equation result (y) vs another (x) across corners."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    
    x_data = results.get(x_sym)
    y_data = results.get(y_sym)
    
    if x_data is None or y_data is None:
        ax.text(0.5, 0.5, f'Data not available for {x_sym} or {y_sym}',
                transform=ax.transAxes, ha='center')
        return ax
    
    # Handle scalar expansion
    if not isinstance(x_data, np.ndarray):
        x_data = np.full_like(y_data, x_data) if isinstance(y_data, np.ndarray) else np.array([x_data])
    if not isinstance(y_data, np.ndarray):
        y_data = np.full_like(x_data, y_data) if isinstance(x_data, np.ndarray) else np.array([y_data])
    
    colors = list(CORNER_COLORS.values())
    
    if x_data.ndim == 2:
        n_corners = x_data.shape[1]
        for c in range(n_corners):
            xc = x_data[:, c]
            yc = y_data[:, c] if y_data.ndim == 2 and c < y_data.shape[1] else y_data.ravel()
            mask = np.isfinite(xc) & np.isfinite(yc)
            if positive_only:
                mask &= (yc > 0)
            ax.plot(xc[mask], yc[mask], color=colors[c % len(colors)],
                    linewidth=1.3, alpha=0.8, label=f'Corner {c+1}')
    elif y_data.ndim == 2:
        n_corners = y_data.shape[1]
        xc = x_data.ravel()
        for c in range(n_corners):
            yc = y_data[:, c]
            mask = np.isfinite(xc) & np.isfinite(yc)
            if positive_only:
                mask &= (yc > 0)
            ax.plot(xc[mask], yc[mask], color=colors[c % len(colors)],
                    linewidth=1.3, alpha=0.8, label=f'Corner {c+1}')
    else:
        mask = np.isfinite(x_data.ravel()) & np.isfinite(y_data.ravel())
        if positive_only:
            mask &= (y_data.ravel() > 0)
        ax.plot(x_data.ravel()[mask], y_data.ravel()[mask], linewidth=1.5)
    
    if constraint_line is not None:
        ax.axhline(y=constraint_line, color='red', linestyle='--', linewidth=1.5,
                   label=constraint_label or f'Spec = {constraint_line:.2e}')
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    if yscale == 'log':
        ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    return ax


fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Current-Mirror OTA — Design Equation Results Across PVT Corners',
             fontsize=14, fontweight='bold')

# Plot drain current i1_2 vs kgm1_2
plot_design_results(results, 'kgm1_2', 'i1_2',
                    'gm/ID of M1,2 [V⁻¹]', 'Drain Current I₁,₂ [A]',
                    'Bias Current vs. gm/ID (Input Pair)',
                    ax=axes[0, 0], yscale='log')

# Plot voltage gain av vs kgm1_2  
plot_design_results(results, 'kgm1_2', 'av',
                    'gm/ID of M1,2 [V⁻¹]', 'Voltage Gain Aᵥ [V/V]',
                    'Voltage Gain vs. gm/ID',
                    ax=axes[0, 1],
                    constraint_line=10**(30/20),
                    constraint_label='Spec: Aᵥ ≥ 30 dB')

# Plot current mirror ratio beta vs kgm1_2
plot_design_results(results, 'kgm1_2', 'beta',
                    'gm/ID of M1,2 [V⁻¹]', 'Mirror Ratio β',
                    'Current Mirror Ratio β vs. gm/ID',
                    ax=axes[1, 0])

# Plot self-loading factor i_sl vs kgm1_2
plot_design_results(results, 'kgm1_2', 'i_sl',
                    'gm/ID of M1,2 [V⁻¹]', 'Self-Loading Factor',
                    'Self-Loading Factor vs. gm/ID',
                    ax=axes[1, 1])

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 5.5 The "Magic Equation" — Optimal gm/ID Selection

ROAR implements the *Id as a function of gm/ID* trade-off curve (referred to internally as the "magic equation"). This curve captures the fundamental **speed vs. power** trade-off: for a given GBW and load capacitance specification, there exists an **optimal gm/ID** that minimizes drain current — the point where increasing gm/ID (moving toward weak inversion) starts to lose more in self-loading than it gains in transconductance efficiency.

In [ ]:
# Use CIDDevice to load all corners and compute the magic equation
nmos_500_device = CIDDevice(
    device_name='n_01v8_L500',
    vdd=1.8,
    lut_directory=os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130',
                               'n_01v8', 'LUT_N_500')
)

# Design specification
GBW_target = 100e6   # 100 MHz
CL_target  = 10e-12  # 10 pF

fig, ax = plt.subplots(figsize=(10, 6))

colors = list(CORNER_COLORS.values())
min_ids_all = []
kgm_opt_all = []

for i, corner in enumerate(nmos_500_device.corners):
    legend = f"{corner.pdk}, L={corner.length}, {corner.corner_name}"
    
    # Compute Id vs kgm using the magic equation
    graph_x, graph_y = [], []
    kgm_col = corner.df['kgm'].values
    kcgd_col = corner.df['kcgd'].values
    kcgs_col = corner.df['kcgs'].values
    kcds_col = corner.df['kcds'].values
    
    min_ids = 1e9
    kgm_opt = 0
    
    for j in range(len(kgm_col)):
        kgm = kgm_col[j]
        kcgd = kcgd_col[j]
        kcgs = kcgs_col[j]
        kcds = kcds_col[j]
        
        strong_inv = 2 * np.pi * GBW_target * CL_target / kgm
        weak_inv = 1 / (1 - 2 * np.pi * GBW_target * (kcgd + kcgs + kcds) / kgm)
        ids = strong_inv * weak_inv
        
        if ids > 0:
            graph_x.append(kgm)
            graph_y.append(ids)
            if ids < min_ids:
                min_ids = ids
                kgm_opt = kgm
    
    ax.semilogy(graph_x, graph_y, color=colors[i % len(colors)],
                linewidth=1.3, label=legend, alpha=0.85)
    min_ids_all.append(min_ids)
    kgm_opt_all.append(kgm_opt)

# Mark the optimal points
for i, (kgm_o, ids_o) in enumerate(zip(kgm_opt_all, min_ids_all)):
    ax.plot(kgm_o, ids_o, 'o', color=colors[i % len(colors)],
            markersize=6, markeredgecolor='black', markeredgewidth=0.5)

ax.set_xlabel('gm/ID [V⁻¹]', fontsize=12)
ax.set_ylabel('Drain Current ID [A]', fontsize=12)
ax.set_title(f'"Magic Equation" — Optimal ID vs. gm/ID\n'
             f'GBW = {GBW_target/1e6:.0f} MHz, C_L = {CL_target*1e12:.0f} pF, '
             f'NMOS n_01v8 L=500 nm',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=7, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 30])
plt.tight_layout()
plt.show()

print(f"\nOptimal operating points across corners:")
print(f"{'Corner':<20} {'gm/ID_opt [V⁻¹]':<18} {'ID_min [µA]':<15}")
print("-" * 53)
for corner, kgm_o, ids_o in zip(nmos_500_device.corners, kgm_opt_all, min_ids_all):
    print(f"{corner.corner_name:<20} {kgm_o:<18.2f} {ids_o*1e6:<15.2f}")

---
## 6. Iterative Solver — Two-Stage Miller OTA

In multi-stage amplifiers, the design equations form **cyclic dependencies**: the output capacitance of Stage 1 depends on the input capacitance of Stage 2 (which is a function of its transistor sizes), while Stage 2's sizing depends on its load — which includes Stage 1's output parasitics.

ROAR's **iterative solver** resolves these cycles using fixed-point iteration with optional damping:

$$x^{(k+1)} = \lambda \cdot f(x^{(k)}) + (1 - \lambda) \cdot x^{(k)}$$

where $\lambda$ is the damping factor (1.0 = no damping).

### 6.1 Cycle Detection with Tarjan's Algorithm

ROAR uses **Tarjan's algorithm** to find strongly-connected components (SCCs) in the dependency graph. Variables in SCCs of size > 1 are the *cycle variables* that need initial guesses.

In [ ]:
# Load the two-stage OTA design
with open(os.path.join(ROAR_HOME, 'design', 'two_stage_ota_iterative.json'), 'r') as f:
    two_stage_design = json.load(f)

print("=" * 70)
print("TWO-STAGE MILLER OTA — Design Equations")
print("=" * 70)
print(f"{'Symbol':<15} {'Expression':<55}")
print("-" * 70)
for eq in two_stage_design['expression_editor']:
    sym = eq['Symbol']
    expr = eq['Expression']
    disabled = '  [DISABLED]' if eq.get('disabled', False) else ''
    print(f"{sym:<15} {expr:<55}{disabled}")

print(f"\n{'='*70}")
print("Constraints")
print("=" * 70)
for c in two_stage_design.get('constraint_editor', []):
    print(f"{c['Symbol']:<15} {c['Constraint Expression']}")

print(f"\n{'='*70}")
print("Instance Table")
print("=" * 70)
for inst in two_stage_design['instance_table']:
    corners_str = ', '.join([c['name'] for c in inst.get('_corners_data', [])])
    model = inst['_corners_data'][0]['model'] if inst.get('_corners_data') else '?'
    length = inst['_corners_data'][0]['length'] if inst.get('_corners_data') else '?'
    print(f"{inst['Instance']:<8} {model:<12} L={length} nm    [{corners_str}]")

In [ ]:
# Build the solver and detect cycles
solver2, corner_dfs2, symbols2 = setup_solver(two_stage_design)

# Detect cycle variables using Tarjan's algorithm
cycle_vars = solver2.find_cycle_variables()

print(f"Dependency graph has {len(solver2.equations)} equations.")
print(f"\nCycle variables detected ({len(cycle_vars)}):")
for var in sorted(cycle_vars):
    print(f"  • {var}")

# Display the dependency graph
dep_graph = solver2.build_dependency_graph()
print(f"\nDependency Graph (equation → depends on):")
print("-" * 50)
for node in sorted(dep_graph.keys()):
    deps = dep_graph[node]
    if deps:
        deps_str = ', '.join(sorted(deps))
        marker = ' ⟲' if node in cycle_vars else ''
        print(f"  {node:<12} → {deps_str}{marker}")

### 6.2 Running the Iterative Solver

In [ ]:
# Get iterative solver settings from the design JSON
iter_settings = two_stage_design.get('iterative_solver', {})
initial_guesses = iter_settings.get('initial_guesses', {})
max_iter = iter_settings.get('max_iterations', 50)
tol = iter_settings.get('tolerance', 1e-6)
damping = iter_settings.get('damping', 1.0)

print("Iterative Solver Settings:")
print(f"  Max iterations: {max_iter}")
print(f"  Tolerance:      {tol}")
print(f"  Damping:        {damping}")
print(f"\nInitial Guesses:")
for var, val in initial_guesses.items():
    print(f"  {var:<12} = {val:.2e}")

# Run the iterative solver
print(f"\n{'='*60}")
print("Running iterative solver...")
print(f"{'='*60}")

results2, info = solver2.evaluate_equations_iterative(
    symbols_to_add=symbols2,
    corner_dfs=corner_dfs2,
    initial_guesses=initial_guesses,
    max_iterations=max_iter,
    tolerance=tol,
    damping=damping
)

# Display convergence results
if info['converged']:
    print(f"\n✓ Converged after {info['iterations']} iteration(s)")
    print(f"  Max relative change: {info['max_rel_change']:.3e}")
else:
    print(f"\n✗ NOT converged after {info['iterations']} iteration(s)")
    print(f"  Max relative change: {info['max_rel_change']:.3e}")
    print("  → Try lowering damping or increasing max iterations.")

print(f"\nCycle variables: {sorted(info['cycle_variables'])}")

In [ ]:
# Display the computed design parameters
if results2 is not None:
    print(f"\n{'='*75}")
    print("Two-Stage OTA — Computed Design Parameters")
    print(f"{'='*75}")
    print(f"{'Symbol':<15} {'Shape':<16} {'Min':<18} {'Max':<18} {'Cycle?'}")
    print("-" * 75)
    for sym in symbols2:
        val = results2.get(sym)
        is_cycle = '⟲' if sym in info['cycle_variables'] else ''
        if val is not None:
            if isinstance(val, np.ndarray):
                shape_str = str(val.shape)
                finite_vals = val[np.isfinite(val)]
                if len(finite_vals) > 0:
                    min_str = f"{np.min(finite_vals):.4e}"
                    max_str = f"{np.max(finite_vals):.4e}"
                else:
                    min_str = max_str = "N/A"
            else:
                shape_str = "scalar"
                min_str = max_str = f"{val:.4e}" if isinstance(val, (int, float)) else str(val)
            print(f"{sym:<15} {shape_str:<16} {min_str:<18} {max_str:<18} {is_cycle}")

In [ ]:
# Visualize the two-stage OTA design trade-offs
if results2 is not None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Two-Stage Miller OTA — Iterative Solver Results (3 PVT Corners)',
                 fontsize=14, fontweight='bold')
    
    # Stage 1 current
    plot_design_results(results2, 'kgm1', 'Id1',
                        'gm/ID of M1 [V⁻¹]', 'Stage 1 Current Id₁ [A]',
                        'Stage 1 Drain Current vs. gm/ID',
                        ax=axes[0, 0], yscale='log')
    
    # Stage 2 current
    plot_design_results(results2, 'kgm5', 'Id5',
                        'gm/ID of M5 [V⁻¹]', 'Stage 2 Current Id₅ [A]',
                        'Stage 2 Drain Current vs. gm/ID',
                        ax=axes[0, 1], yscale='log')
    
    # Total gain (Av1 × Av2)
    plot_design_results(results2, 'kgm1', 'Av_total',
                        'gm/ID of M1 [V⁻¹]', 'Total Gain Av₁·Av₂ [V/V]',
                        'Total Open-Loop Gain',
                        ax=axes[1, 0], yscale='log',
                        constraint_line=1000,
                        constraint_label='Spec: Av > 60 dB (1000 V/V)')
    
    # Total current
    plot_design_results(results2, 'kgm1', 'Itotal',
                        'gm/ID of M1 [V⁻¹]', 'Total Current [A]',
                        'Total Power Budget',
                        ax=axes[1, 1], yscale='log',
                        constraint_line=500e-6,
                        constraint_label='Spec: Itotal < 500 µA')
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

### 6.3 Convergence of the Iterative Solver

To demonstrate the convergence behavior, we run the iterative solver step-by-step and track how the cycle variables evolve.

In [ ]:
# Track convergence by running the solver with increasing max_iterations
convergence_history = {var: [] for var in cycle_vars}
rel_change_history = []

for max_i in range(1, 21):
    solver_conv, corner_dfs_conv, symbols_conv = setup_solver(two_stage_design)
    res, inf = solver_conv.evaluate_equations_iterative(
        symbols_to_add=symbols_conv,
        corner_dfs=corner_dfs_conv,
        initial_guesses=initial_guesses,
        max_iterations=max_i,
        tolerance=1e-15,  # Very tight so it doesn't stop early
        damping=1.0
    )
    rel_change_history.append(inf['max_rel_change'])
    if res is not None:
        for var in cycle_vars:
            val = res.get(var)
            if val is not None:
                if isinstance(val, np.ndarray):
                    convergence_history[var].append(np.mean(val[np.isfinite(val)]))
                else:
                    convergence_history[var].append(float(val))
            else:
                convergence_history[var].append(np.nan)

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Relative change per iteration
ax1.semilogy(range(1, len(rel_change_history) + 1), rel_change_history, 'b-o', linewidth=2)
ax1.axhline(y=tol, color='red', linestyle='--', label=f'Tolerance = {tol}')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Max Relative Change')
ax1.set_title('Iterative Solver Convergence', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Cycle variable evolution
for var in sorted(cycle_vars):
    vals = convergence_history[var]
    if vals and not all(np.isnan(v) for v in vals):
        ax2.plot(range(1, len(vals) + 1), vals, '-o', label=var, linewidth=1.5, markersize=4)

ax2.set_xlabel('Iteration')
ax2.set_ylabel('Value (Mean Across Corners)')
ax2.set_title('Cycle Variable Evolution', fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. SPICE Netlist Generation

ROAR can generate SPICE netlists from the designed circuits. Below is the output netlist for the current-mirror OTA, ready for simulation in ngspice or any SPICE-compatible simulator.

In [ ]:
# Display the generated SPICE netlist
spice_path = os.path.join(ROAR_HOME, 'design', 'cm_ota.spice')
with open(spice_path, 'r') as f:
    netlist = f.read()

print("=" * 70)
print("Generated SPICE Netlist — Current-Mirror OTA (SKY130)")
print("=" * 70)
print(netlist)
print("=" * 70)
print("\nKey observations:")
print("  • Transistor widths (w1_2, w3_4, etc.) are parameterized")
print("  • ROAR computes these from the design equations: W = ID / (ID/W)")
print("  • The designer selects the operating gm/ID from the trade-off curves")
print("  • All transistors use L = 500 nm (sky130_fd_pr__nfet_01v8 / pfet_01v8)")

---
## 8. Multi-Corner Verification Results

The current-mirror OTA was simulated post-layout (GDS) across three temperature corners. The images below show ROAR's Bode plot results.

### 8.1 GDS Simulation Results

In [ ]:
from IPython.display import Image, display, HTML

# Display GDS simulation results
gds_images = [
    ('cm_ota_-25c_gds.PNG', 'GDS Simulation: T = −25 °C'),
    ('cm_ota_25c_gds.PNG',  'GDS Simulation: T = 25 °C'),
    ('cm_ota_75c_gds.PNG',  'GDS Simulation: T = 75 °C'),
]

for img_file, caption in gds_images:
    img_path = os.path.join(ROAR_HOME, 'images', 'png', img_file)
    if os.path.exists(img_path):
        display(HTML(f'<h4>{caption}</h4>'))
        display(Image(filename=img_path, width=700))
    else:
        print(f"Image not found: {img_path}")

In [ ]:
# Show the combined all-corners result
all_corners_img = os.path.join(ROAR_HOME, 'images', 'png', 'cm_ota_all_gds.PNG')
if os.path.exists(all_corners_img):
    display(HTML('<h4>GDS Simulation: All Temperature Corners Combined</h4>'))
    display(Image(filename=all_corners_img, width=700))

---
## 9. PDK Portability Demonstration

ROAR's key differentiator is **PDK agnosticism**. The same design methodology, equations, and GUI work with any process technology — only the LUTs change. Let's demonstrate by comparing device characteristics across available PDKs.

In [ ]:
# Check what PDKs are available
tech_list_path = os.path.join(ROAR_HOME, 'tech_list.txt')
print("Registered PDKs in tech_list.txt:")
print("=" * 50)
with open(tech_list_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith('#'):
            print(f"  {line}")

# Compare SKY130 NMOS vs IHP SG13G2 NMOS (if available)
ihp_lut_base = os.path.join(ROAR_HOME, 'characterization', 'ihp130', 'LUTs_IHP130')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# SKY130 NMOS TT 25C, L=150nm
sky_path = os.path.join(ROAR_HOME, 'characterization', 'sky130', 'LUTs_SKY130',
                        'n_01v8', 'LUT_N_150', 'nfettt25.csv')
if os.path.exists(sky_path):
    sky_corner = CIDCorner(corner_name='SKY130_tt25', lut_csv=sky_path, vdd=1.8)
    df_sky = sky_corner.df
    mask = (df_sky['kgm'] > 0.5) & (df_sky['kgm'] < 30)
    ax1.semilogy(df_sky.loc[mask, 'kgm'], df_sky.loc[mask, 'ft'],
                 'b-', linewidth=2, label=f'SKY130 n_01v8 L={sky_corner.length}')
    ax2.semilogy(df_sky.loc[mask, 'kgm'], df_sky.loc[mask, 'gmro'],
                 'b-', linewidth=2, label=f'SKY130 n_01v8 L={sky_corner.length}')

# Try IHP NMOS
if os.path.exists(ihp_lut_base):
    # Find an NMOS model directory
    for model_dir in os.listdir(ihp_lut_base):
        if 'nfet' in model_dir.lower() or 'n' == model_dir[0].lower():
            model_path = os.path.join(ihp_lut_base, model_dir)
            if os.path.isdir(model_path):
                # Find first length directory
                for length_dir in sorted(os.listdir(model_path)):
                    length_path = os.path.join(model_path, length_dir)
                    if os.path.isdir(length_path):
                        # Find a TT corner
                        for csv_file in os.listdir(length_path):
                            if 'tt' in csv_file.lower() and csv_file.endswith('.csv'):
                                ihp_path = os.path.join(length_path, csv_file)
                                ihp_corner = CIDCorner(corner_name=f'IHP_{csv_file}',
                                                        lut_csv=ihp_path, vdd=1.2)
                                df_ihp = ihp_corner.df
                                mask_ihp = (df_ihp['kgm'] > 0.5) & (df_ihp['kgm'] < 30)
                                ax1.semilogy(df_ihp.loc[mask_ihp, 'kgm'],
                                             df_ihp.loc[mask_ihp, 'ft'],
                                             'r--', linewidth=2,
                                             label=f'IHP-SG13G2 {model_dir} L={ihp_corner.length}')
                                ax2.semilogy(df_ihp.loc[mask_ihp, 'kgm'],
                                             df_ihp.loc[mask_ihp, 'gmro'],
                                             'r--', linewidth=2,
                                             label=f'IHP-SG13G2 {model_dir} L={ihp_corner.length}')
                                break
                        break
            break

for ax, ylabel, title in [
    (ax1, 'fₜ [Hz]', 'Transit Frequency Comparison'),
    (ax2, 'gm·rₒ [V/V]', 'Intrinsic Gain Comparison'),
]:
    ax.set_xlabel('gm/ID [V⁻¹]')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Cross-PDK Device Comparison — ROAR LUT Framework',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

---
## 10. ROAR GUI Overview

While this notebook demonstrates ROAR's capabilities programmatically, the full ROAR application provides a rich **PyQt6-based GUI** with:

- **Tech Browser** — Hierarchical tree for navigating PDKs, models, lengths, and corners  
- **Lookup Windows** — Real-time interactive plots of device and design-equation curves  
- **Design Editor** — Expression editor, constraint editor, and instance table  
- **Iterative Solver Dialog** — Automatic cycle detection and convergence monitoring  
- **SPICE Netlist Generator** — Export sized circuits for simulation  
- **Python Console** — Built-in REPL for scripting within the application  

### Screenshots

In [ ]:
# Display available GUI screenshots
gui_images = [
    ('fig_four_graphs_small.png', 'ROAR GUI — Four Canonical gm/ID Plots'),
    ('fig_bode_plot.png', 'ROAR GUI — Bode Plot Visualization'),
]

for img_file, caption in gui_images:
    img_path = os.path.join(ROAR_HOME, 'images', 'png', img_file)
    if os.path.exists(img_path):
        display(HTML(f'<h4>{caption}</h4>'))
        display(Image(filename=img_path, width=700))
    else:
        print(f"Image not found: {img_file}")

---
## 11. Summary and Conclusions

### What ROAR Provides

| Feature | Description |
|---------|-------------|
| **PDK-Agnostic Design** | Works with any process via CSV lookup tables |
| **gm/ID Methodology** | Unified weak-to-strong inversion design framework |
| **Multi-Corner Analysis** | Simultaneous evaluation across 9+ PVT corners |
| **Symbolic Expressions** | SymPy-based equation engine with topological sorting |
| **Iterative Solver** | Tarjan's SCC detection + fixed-point iteration with damping |
| **Interactive GUI** | PyQt6 app with real-time plotting and tech browser |
| **SPICE Generation** | Automated netlist export for simulation verification |
| **Open-Source PDKs** | Ships with SKY130 and IHP SG13G2 LUTs |

### Design Flow Summary

```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│  Characterize    │    │   Design in ROAR  │    │   Verify in      │
│  (ngspice/       │ →  │   (gm/ID curves,  │ →  │   SPICE           │
│   Spectre/HSPICE)│    │    equations,      │    │   (ngspice/       │
│                  │    │    constraints)     │    │    Spectre)       │
│  → CSV LUTs      │    │  → Sized netlist   │    │  → Layout (GDS)  │
└─────────────────┘    └──────────────────┘    └─────────────────┘
```

### Demonstrated in This Notebook

1. ✅ Loaded and inspected SKY130 lookup tables (325 bias points × 9 PVT corners)  
2. ✅ Visualized the four canonical gm/ID design curves across all corners  
3. ✅ Analyzed process variation impact on fₜ and gm·rₒ  
4. ✅ Compared device characteristics across channel lengths  
5. ✅ Designed a current-mirror OTA using ROAR's equation solver  
6. ✅ Demonstrated the iterative solver on a two-stage Miller OTA with cyclic dependencies  
7. ✅ Tracked solver convergence using Tarjan's cycle detection  
8. ✅ Showed GDS-level post-layout simulation results  
9. ✅ Demonstrated cross-PDK portability (SKY130 vs. IHP SG13G2)  

### Future Work

- **Layout automation integration** (ALIGN, Magic)  
- **Optimization engine** — automated gm/ID selection for multi-objective targets  
- **Additional PDK support** — GF180, TSMC N7 (with appropriate NDAs)  
- **Monte Carlo aware design** — statistical corner analysis  
- **Hierarchical design** — reusable block-level templates  

---

*ROAR is open-source software developed by Alec S. Adair.*  
*Submitted to the IEEE SSCS Code-a-Chip Challenge, 2026.*

---
## References

1. P. Jespers and B. Murmann, *Systematic Design of Analog CMOS Circuits Using Pre-Computed Lookup Tables*, Cambridge University Press, 2017.  
2. B. Murmann, "The gm/ID Design Methodology Revisited," *IEEE Solid-State Circuits Magazine*, 2023.  
3. F. Silveira, D. Flandre, and P. G. A. Jespers, "A gm/ID based methodology for the design of CMOS analog circuits and its application to the synthesis of a Silicon-on-Insulator Micropower OTA," *IEEE J. Solid-State Circuits*, vol. 31, no. 9, pp. 1314–1319, Sep. 1996.  
4. T. Edwards et al., "SkyWater SKY130 Open Source PDK," Google/SkyWater Technology Foundry, 2020. [Online]. Available: https://github.com/google/skywater-pdk  
5. IHP Microelectronics, "IHP Open Source PDK SG13G2," 2024. [Online]. Available: https://github.com/IHP-GmbH/IHP-Open-PDK  